# Advanced AI Image Humanization: Spectral Artifact Removal using Modified U-Net

This notebook implements an advanced "digital forensic" architecture to remove frequency artifacts (spectral fingerprints) from AI-generated images.

**Key Components:**
1.  **Modified U-Net Generator**:
    *   Replaces standard `Transposed Convolution` with `Bilinear Upsampling` + `Convolution` to avoid checkerboard artifacts.
    *   Integrates **Feature Scaling Layers** (Instance Norm/Scaling) to stabilize reconstruction.
2.  **Spectral Discriminator**:
    *   Operates on the **2D-DFT Magnitude Spectrum** of the image to penalize unnatural frequency patterns.
3.  **Perceptual Loss (VGG16)**:
    *   Ensures the "humanized" image retains the visual content of the original input.

**References:**
*   *Analisis Penilaian Kualitas Citra Tanpa Referensi...*
*   *Arsitektur Forensik Digital Lanjut: Integrasi Humanisasi Citra Sintetik...*

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os

# Device Configuration
device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Random Seed for Reproducibility
torch.manual_seed(42)
np.random.seed(42)

Using device: mps


## 1. Feature Scaling & Modified Upsampling Block

As per the requirements, we implement a custom block that avoids `ConvTranspose2d` artifacts.
*   **Upsampling**: `Bilinear` mode.
*   **Convolution**: Refines features after upsampling.
*   **Feature Scaling/Norm**: `InstanceNorm2d` is used here to normalize feature maps, acting as the critical "scaling" component to prevent noise introduction.

In [ ]:
class FeatureScalingLayer(nn.Module):
    """
    Custom Feature Scaling Layer.
    Ideally, this ensures that the distribution of features remains consistent
    after the upsampling operation to prevent 'checkerboard' artifacts.
    """
    def __init__(self, channels):
        super(FeatureScalingLayer, self).__init__()
        # Instance Normalization acts as a feature scaler per sample, widely used in style transfer/generative tasks.
        self.norm = nn.InstanceNorm2d(channels, affine=True)

    def forward(self, x):
        return self.norm(x)

class ModifiedUpsampleBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ModifiedUpsampleBlock, self).__init__()
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.scaling = FeatureScalingLayer(out_channels) # Critical component
        self.activation = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        x = self.upsample(x)
        x = self.conv(x)
        x = self.scaling(x)
        x = self.activation(x)
        return x

## 2. Modified U-Net Generator Architecture

This Generator takes an image (presumably with AI artifacts) and outputs a "clean" image.
*   **Encoder**: Standard downsampling path to capture context.
*   **Decoder**: Uses `ModifiedUpsampleBlock` to reconstruct high-frequency details without grid artifacts.

In [ ]:
class UNetGenerator(nn.Module):
    def __init__(self, in_channels=3, out_channels=3):
        super(UNetGenerator, self).__init__()

        # Encoder (Downsampling)
        self.enc1 = self.conv_block(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = self.conv_block(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = self.conv_block(128, 256)
        self.pool3 = nn.MaxPool2d(2)
        self.enc4 = self.conv_block(256, 512)
        self.pool4 = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = self.conv_block(512, 1024)

        # Decoder (Upsampling with Modified Blocks)
        self.up4 = ModifiedUpsampleBlock(1024, 512)
        self.dec4 = self.conv_block(1024, 512) # 512 from up, 512 from skip
        
        self.up3 = ModifiedUpsampleBlock(512, 256)
        self.dec3 = self.conv_block(512, 256)

        self.up2 = ModifiedUpsampleBlock(256, 128)
        self.dec2 = self.conv_block(256, 128)

        self.up1 = ModifiedUpsampleBlock(128, 64)
        self.dec1 = self.conv_block(128, 64)

        # Final Output
        self.final = nn.Conv2d(64, out_channels, kernel_size=1)
        self.tanh = nn.Tanh() # Assuming input images are normalized to [-1, 1]

    def conv_block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.LeakyReLU(0.2, inplace=True)
        )

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))
        
        # Bottleneck
        b = self.bottleneck(self.pool4(e4))

        # Decoder with Skip Connections
        d4 = self.up4(b)
        # Handle potential size mismatch due to padding/pooling
        if d4.size() != e4.size():
            d4 = torch.nn.functional.interpolate(d4, size=e4.shape[2:])
        d4 = torch.cat((d4, e4), dim=1)
        d4 = self.dec4(d4)

        d3 = self.up3(d4)
        if d3.size() != e3.size():
            d3 = torch.nn.functional.interpolate(d3, size=e3.shape[2:])
        d3 = torch.cat((d3, e3), dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        if d2.size() != e2.size():
            d2 = torch.nn.functional.interpolate(d2, size=e2.shape[2:])
        d2 = torch.cat((d2, e2), dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        if d1.size() != e1.size():
            d1 = torch.nn.functional.interpolate(d1, size=e1.shape[2:])
        d1 = torch.cat((d1, e1), dim=1)
        d1 = self.dec1(d1)

        return self.tanh(self.final(d1))

## 3. Spectral Discriminator

This discriminator works on the **Frequency Domain**.
1.  Compute 2D-DFT.
2.  Take the Log-Magnitude Spectrum.
3.  Classify based on spectral patterns (Real vs. AI).

In [ ]:
class SpectralDiscriminator(nn.Module):
    def __init__(self, in_channels=3):
        super(SpectralDiscriminator, self).__init__()
        
        # Input: Log magnitude spectrum is often 1 channel (grayscale) or 3 channels depending on implementation.
        # We will assume we process the magnitude spectrum of the 3 channels.
        
        self.main = nn.Sequential(
            # Frequencies often have high dynamic range, but log-magnitude reduces it.
            # Assuming input size [Batch, 3, H, W] representing the spectrum.
            
            nn.Conv2d(in_channels, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(256, 512, kernel_size=4, stride=1, padding=1),
            nn.InstanceNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(512, 1, kernel_size=4, stride=1, padding=1),
            # No Sigmoid here if using BCEWithLogitsLoss (recommended for stability)
        )

    def forward(self, x):
        # x is expected to be the image
        # We first convert to frequency domain inside the discriminator or pass pre-processed spectra.
        # Let's handle conversion here for convenience.
        
        # 1. FFT
        fft_x = torch.fft.rfft2(x, norm='ortho')
        
        # 2. Magnitude Spectrum
        # Add epsilon to avoid log(0)
        epsilon = 1e-8
        mag_x = torch.abs(fft_x)
        log_mag_x = torch.log(mag_x + epsilon)
        
        # Normalize roughly to [-1, 1] range for better training stability (optional but good practice)
        # Simple instance-based scaling
        # log_mag_x = (log_mag_x - log_mag_x.mean(dim=(2,3), keepdim=True)) / (log_mag_x.std(dim=(2,3), keepdim=True) + epsilon)

        return self.main(log_mag_x)

## 4. Perceptual Loss (VGG16)

Instead of just checking if pixels match (MSE), we check if the **content** matches by looking at features via a pre-trained VGG16 network. This prevents the "cleaning" process from blurring the image.

In [ ]:
class PerceptualLoss(nn.Module):
    def __init__(self):
        super(PerceptualLoss, self).__init__()
        # Load VGG16 pretrained on ImageNet
        vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        
        # We only need the feature extractor, not the classifier
        # We usually take features up to ReLU3_3 or similar.
        # VGG16 features: 
        # Slice 1: relu1_2 (index 3)
        # Slice 2: relu2_2 (index 8)
        # Slice 3: relu3_3 (index 15)
        # Slice 4: relu4_3 (index 22)
        
        self.slice1 = nn.Sequential(*list(vgg.features)[:4]).eval()
        self.slice2 = nn.Sequential(*list(vgg.features)[4:9]).eval()
        self.slice3 = nn.Sequential(*list(vgg.features)[9:16]).eval()
        
        # Freeze parameters
        for param in self.parameters():
            param.requires_grad = False
            
    def forward(self, x, y):
        # x: input image, y: target image
        # Normalize inputs for VGG if necessary (assuming roughly [0,1] or similar scale)
        # Ideally, we should apply ImageNet normalization here if validation is strict.
        
        x_vgg = x
        y_vgg = y
        
        loss = 0.0
        
        f_x = self.slice1(x_vgg)
        f_y = self.slice1(y_vgg)
        loss += torch.mean((f_x - f_y) ** 2)
        
        f_x = self.slice2(f_x)
        f_y = self.slice2(f_y)
        loss += torch.mean((f_x - f_y) ** 2)
        
        f_x = self.slice3(f_x)
        f_y = self.slice3(f_y)
        loss += torch.mean((f_x - f_y) ** 2)
        
        return loss

## 5. Training Setup & Dummy Verification

Here we simulate the training loop.
1.  **Initialize Models**: Generator and Discriminator.
2.  **Optimizers**: Adam is standard for GANs.
3.  **Dummy Train Step**: Run one forward/backward pass with random tensors to ensure shapes and gradients flow correctly.

In [ ]:
# 1. Initialize Models
generator = UNetGenerator().to(device)
discriminator = SpectralDiscriminator().to(device)
perceptual_criterion = PerceptualLoss().to(device)

# 2. Key Optimizers
# Beta1=0.5 is standard for GAN stability
optimizer_G = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

# Standard GAN Loss (BCE)
adversarial_criterion = nn.BCEWithLogitsLoss()

# 3. Simple FFT Visualization Helper
def plot_spectrum(img_tensor, title="Spectrum"):
    """
    Plots the log-magnitude spectrum of an image tensor.
    img_tensor: [C, H, W] (Single image, not batch)
    """
    img_tensor = img_tensor.cpu()
    # Convert to grayscale for spectrum analysis if RGB
    if img_tensor.shape[0] == 3:
        img_gray = 0.299 * img_tensor[0] + 0.587 * img_tensor[1] + 0.114 * img_tensor[2]
    else:
        img_gray = img_tensor[0]
        
    f = torch.fft.fft2(img_gray)
    fshift = torch.fft.fftshift(f)
    magnitude_spectrum = 20 * torch.log(torch.abs(fshift) + 1e-8)
    
    plt.imshow(magnitude_spectrum.numpy(), cmap='gray')
    plt.title(title)
    plt.axis('off')

# 4. Dummy Training Loop Verification
print("Starting Dummy Training Step...")

# Simulation: Batch of 4 images, 3 channels, 256x256
real_images = torch.randn(4, 3, 256, 256).to(device)
ai_generated_images = torch.randn(4, 3, 256, 256).to(device) # In reality, these would have artifacts

# --- Train Discriminator ---
optimizer_D.zero_grad()

# Real images (Label = 1)
# Note: In a real scenario, we might not have "Real" images corresponding perfectly to AI ones if unpaired.
# But for GANs, we need a set of Real images to teach the discriminator what the spectrum SHOULD look like.
pred_real = discriminator(real_images)
loss_real = adversarial_criterion(pred_real, torch.ones_like(pred_real))

# Fake (Humanized) images
fake_images = generator(ai_generated_images)
pred_fake = discriminator(fake_images.detach()) # Detach to avoid G gradients
loss_fake = adversarial_criterion(pred_fake, torch.zeros_like(pred_fake))

loss_D = (loss_real + loss_fake) / 2
loss_D.backward()
optimizer_D.step()

# --- Train Generator ---
optimizer_G.zero_grad()

# Adversarial Loss (We want D to think these are Real, so Label = 1)
pred_fake_for_G = discriminator(fake_images) # No detach here
loss_G_adv = adversarial_criterion(pred_fake_for_G, torch.ones_like(pred_fake_for_G))

# Perceptual Loss (Content Preservation)
# We want the output to look like the input content-wise, just without artifacts.
loss_G_perceptual = perceptual_criterion(fake_images, ai_generated_images)

# Total G Loss
# Weighted sum (lambda parameters would be tuned)
lambda_adv = 0.1
lambda_perc = 1.0
loss_G = (lambda_adv * loss_G_adv) + (lambda_perc * loss_G_perceptual)

loss_G.backward()
optimizer_G.step()

print("Dummy training step completed successfully!")
print(f"Loss D: {loss_D.item():.4f}")
print(f"Loss G: {loss_G.item():.4f} (Adv: {loss_G_adv.item():.4f}, Perc: {loss_G_perceptual.item():.4f})")

# Visualize Result
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plot_spectrum(ai_generated_images[0], "Original AI Spectrum (Random Noise)")
plt.subplot(1, 2, 2)
plot_spectrum(fake_images[0].detach(), "Humanized Spectrum")
plt.show()

## 6. Complete Workflow Implementation (Inference Demo)

Bagian ini mengimplementasikan alur kerja praktis sesuai requirement:
1.  **Input**: Load citra asli.
2.  **Visualisasi Awal**: Citra + FFT Spectrum.
3.  **Proses**: Feed-forward ke Generator.
4.  **Evaluasi**: Bandingkan hasil (Citra Humanis + Spectrum-nya).
5.  **Output**: Simpan hasil.

In [ ]:
def visualize_analysis(image_tensor, title="Image"):
    """
    Menampilkan Citra Spasial & Spektrum Frekuensi berdampingan.
    input: image_tensor [C, H, W]
    """
    img_tensor = image_tensor.cpu().detach()
    
    # 1. Prepare Spatial Image for Display
    # Denormalize if trained with [-1, 1], here assuming raw output or [0,1]
    # For visualization, we clamp to [0,1]
    img_disp = img_tensor.permute(1, 2, 0).numpy()
    img_disp = (img_disp - img_disp.min()) / (img_disp.max() - img_disp.min())
    
    # 2. Compute Spectrum (FFT)
    if img_tensor.shape[0] == 3:
        img_gray = 0.299 * img_tensor[0] + 0.587 * img_tensor[1] + 0.114 * img_tensor[2]
    else:
        img_gray = img_tensor[0]
        
    f = torch.fft.fft2(img_gray)
    fshift = torch.fft.fftshift(f)
    magnitude_spectrum = 20 * torch.log(torch.abs(fshift) + 1e-8)
    
    # Plotting
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Spatial
    axes[0].imshow(img_disp)
    axes[0].set_title(f"{title} (Spatial)")
    axes[0].axis('off')
    
    # Spectral
    axes[1].imshow(magnitude_spectrum.numpy(), cmap='gray')
    axes[1].set_title(f"{title} (Frequency Spectrum)")
    axes[1].axis('off')
    
    plt.show()

def humanize_image_pipeline(image_path, model, save_path="output_humanized.png"):
    # 1. Input: Load Image
    try:
        img_pil = Image.open(image_path).convert('RGB')
    except Exception as e:
        print(f"Error loading image: {e}")
        print("Using dummy noise image for demonstration.")
        img_pil = Image.fromarray(np.random.randint(0, 255, (256, 256, 3), dtype=np.uint8))
    
    # Preprocessing
    transform = transforms.Compose([
        transforms.Resize((256, 256)), # Resize for model
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # Normalize to [-1, 1]
    ])
    
    img_tensor = transform(img_pil).unsqueeze(0).to(device) # Add batch dim
    
    # 2. Visualisasi Awal (Input Analysis)
    print("--- Step 1 & 2: Input & Analisis Awal ---")
    visualize_analysis(img_tensor[0], title="Input (AI Artifacts)")
    
    # 3. Proses Penghapusan (Inference)
    model.eval()
    with torch.no_grad():
        # Advanced Path: Through U-Net Generator
        humanized_tensor = model(img_tensor)
    
    # 4. Evaluasi (Result Analysis)
    print("--- Step 3 & 4: Proses & Evaluasi ---")
    visualize_analysis(humanized_tensor[0], title="Output (Humanized)")
    
    # 5. Output: Save Image
    output_img = humanized_tensor[0].cpu().permute(1, 2, 0).numpy()
    # Denormalize from [-1, 1] to [0, 255]
    output_img = ((output_img * 0.5 + 0.5) * 255).astype(np.uint8)
    output_pil = Image.fromarray(output_img)
    output_pil.save(save_path)
    print(f"--- Step 5: Saved result to {save_path} ---")

# --- Jalankan Demo Alur Kerja ---
# Buat file dummy 'synthetic_sample.png' jika belum ada untuk demo
if not os.path.exists('synthetic_sample.png'):
    dummy_data = np.random.randint(0, 255, (256, 256, 3), dtype=np.uint8)
    Image.fromarray(dummy_data).save('synthetic_sample.png')

# Jalankan pipeline
humanize_image_pipeline('synthetic_sample.png', generator)